# Python for (open) Neuroscience  
### Practical 2.3 — *Pandas foundations*
#### Module 02 - Scientific Stack

**How to use this notebook**
- Complete the `____` blanks.
- Keep the starter code structure whenever possible.
- Run each code cell with **Shift+Enter**


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

np.random.seed(42)


## Shared dataset

Run the next cell first.  
We will use this DataFrame in several exercises.

It mimics a small cognitive neuroscience experiment with:
- participant-level variables
- behavioral performance
- one missing value in `accuracy`
- one missing value in `roi_signal`



In [ ]:
df = pd.DataFrame({
    "participant_id": ["sub-01", "sub-02", "sub-03", "sub-04", "sub-05", "sub-06", "sub-07", "sub-08"],
    "group": ["control", "control", "patient", "patient", "control", "patient", "control", "patient"],
    "age": [24, 29, 31, 27, 22, 35, 26, 30],
    "task": ["memory", "attention", "memory", "attention", "memory", "memory", "attention", "memory"],
    "rt_ms": [612, 540, 705, 498, 580, 690, 520, 640],
    "accuracy": [0.91, 0.95, 0.78, 0.88, np.nan, 0.81, 0.93, 0.85],
    "roi_signal": [1.20, 0.98, 1.45, 1.05, 1.10, np.nan, 0.95, 1.32],
    "session": [1, 1, 1, 2, 2, 2, 1, 2]
})

df


## 1. First inspection

In [ ]:
# Task:
# 1) Show the first 3 rows.
# 2) Print the dataframe shape.
# 3) Display the column names.
# 4) Display the data types.

print(df.head(3))
print("shape:", df.shape)
print("columns:", df.columns)
df.dtypes

## 2. Select columns and check object types

In [ ]:
# Task:
# 1) Select the "age" column as a Series.
# 2) Select ["participant_id", "task", "rt_ms"] as a DataFrame.
# 3) Print the type of each result.

age_series = df["age"]
subset_df = df[["participant_id", "task", "rt_ms"]]

print(type(age_series))
print(type(subset_df))

age_series

## 3. Boolean masks with multiple conditions

In [ ]:
# Task:
# Keep only participants who:
# - performed the "memory" task
# - AND have rt_ms above 600

selection = (df["task"] == "memory") & (df["rt_ms"] > 600)

filtered = df[selection]
filtered


## 4. `.loc` versus `.iloc`

Remember:
- `.loc[row_selection, column_selection]` uses **labels**
- `.iloc[row_selection, column_selection]` uses **integer positions**



In [ ]:
# Task:
# 1) Using .loc, select rows with labels 1 to 4 and columns ["participant_id", "accuracy"].
# 2) Using .iloc, select the first 5 rows and the first 3 columns.

with_loc = df.loc[1:4, ["participant_id", "accuracy"]]
with_iloc = df.iloc[:5,:3]

print(with_loc)
print()
print(with_iloc)

## 5. Set a meaningful row index

In [ ]:
# Task:
# 1) Create a copy of df called indexed_df.
# 2) Use participant_id as the row index.
# 3) With .loc, select rows from sub-02 to sub-05
#    and only the columns ["group", "rt_ms", "accuracy"].

indexed_df = df.copy()
indexed_df = indexed_df.set_index("participant_id")

indexed_df.loc["sub-02":"sub-05", ["group","rt_ms","accuracy"]]


## 6. Rank participants by performance

`sort_values()` sorts a DataFrame by one or more columns.

`assign()` returns a new DataFrame with new columns added.

Both are useful when you want a clean pipeline instead of editing the original table in place.


In [ ]:
# Task:
# 1) Create a new column called "efficiency" defined as accuracy / rt_ms.
# 2) Sort the result by "efficiency" in descending order.
# 3) Show only participant_id, task, accuracy, rt_ms, and efficiency.

ranked = (
    df
    .assign(efficiency=lambda data: data["accuracy"] / data["rt_ms"])
    .sort_values(by="efficiency", ascending=False)
)

ranked[["participant_id","task","accuracy","rt_ms","efficiency"]]

## 7. Use `isin()` and `between()`

In [ ]:
# Task:
# Keep rows where:
# - group is either "control" or "patient"   (use isin, even if that includes all rows)
# - age is between 25 and 31 inclusive
# - session is equal to 1

mask = (
    df["group"].isin(["control","patient"])
    & df["age"].between(25, 31)
    & (df["session"] == 1)
)

df[mask]


## 8. Missing values

Useful reminders:
- `.isna()` marks missing values with `True`
- `.sum()` on a boolean Series counts how many `True` values there are
- many summary methods ignore missing values by default



In [ ]:
# Task:
# 1) Count missing values in each column.
# 2) Compute the mean accuracy.
# 3) Compute the mean roi_signal.
missing_per_column = df.isna().sum()
mean_accuracy = df["accuracy"].mean()
mean_roi = df["roi_signal"].mean()

print(missing_per_column)
print("mean accuracy:", mean_accuracy)
print("mean roi signal:", mean_roi)



## 9. Fill missing values with a column mean

`fillna(value)` replaces missing values.

A common pattern is:
- compute a summary value from a column
- use it to fill missing entries in that same column



In [ ]:
# Task:
# 1) Create df_filled as a copy of df.
# 2) Replace missing values in "accuracy" with the mean accuracy.
# 3) Replace missing values in "roi_signal" with the mean roi_signal.
# 4) Verify that there are now zero missing values in those two columns.

df_filled = df.copy()

acc_mean = df_filled["accuracy"].mean()
roi_mean = df_filled["roi_signal"].mean()

df_filled["accuracy"] = df_filled["accuracy"].fillna(acc_mean)
df_filled["roi_signal"] = df_filled["roi_signal"].fillna(roi_mean)

df_filled[["accuracy","roi_signal"]].isna().sum()


## 10. Find extreme rows with `idxmax()` and `idxmin()`

In pandas:
- `.idxmax()` returns the **row label** where a Series reaches its maximum
- `.idxmin()` returns the **row label** where a Series reaches its minimum**

This is closely related to NumPy's `argmax()` and `argmin()`, but pandas returns the label rather than the integer position.


In [ ]:
# Task:
# 1) Find the row label of the fastest participant (minimum rt_ms).
# 2) Find the row label of the participant with the largest roi_signal.
# 3) Use those labels with .loc to show the corresponding rows.

fastest_idx = df["rt_ms"].idxmin()
strongest_roi_idx = df["roi_signal"].idxmax()

print("fastest row label:", fastest_idx)
print("strongest ROI row label:", strongest_roi_idx)

df.loc[[fastest_idx,strongest_roi_idx]]


## 11. Group summaries with `groupby()`

`groupby()` splits the DataFrame into groups, applies a computation to each group, and combines the results.

This is one of the most important pandas tools for real analyses.


In [ ]:
# Task:
# Compute the mean reaction time and mean accuracy separately for each group
# ("control" vs "patient").

group_summary = df.groupby("group")[["rt_ms","accuracy"]].mean()
group_summary


## 12. Count participants by task and session

`pd.crosstab()` creates a frequency table.

It is handy for checking whether an experiment is balanced across conditions.


In [ ]:
# Task:
# Build a table with:
# - rows = task
# - columns = session
# - values = participant counts

task_session_counts = pd.crosstab(df["task"],df["session"])
task_session_counts


## 13. Read a CSV file

`pd.read_csv(path)` reads a comma-separated text file into a DataFrame.

In real workflows, pandas often starts here.


In [ ]:
# Run this cell once to create a tiny CSV file in the current folder.
csv_path = Path("neuro_behavior.csv")

df_small = pd.DataFrame({
    "participant_id": ["sub-09", "sub-10", "sub-11", "sub-12"],
    "condition": ["faces", "houses", "faces", "houses"],
    "trial_count": [120, 115, 128, 118],
    "mean_rt_ms": [510, 545, 498, 552]
})

df_small.to_csv(csv_path, index=False)
print(csv_path.resolve())

In [ ]:
# Task:
# 1) Read the CSV file into a DataFrame called loaded_df.
# 2) Show the first rows.
# 3) Print the shape.

loaded_df = pd.read_csv(csv_path)
print(loaded_df.head())
print(loaded_df.shape)


## 14. From pandas to NumPy: row-wise centering with `keepdims=True`

`keepdims=True` is a **NumPy** argument, not a pandas method.

When reducing an array across an axis:
- without `keepdims`, the reduced axis disappears
- with `keepdims=True`, the reduced axis is kept with length 1

This is useful for broadcasting, for example when subtracting each row mean from the original matrix.


In [ ]:
# Suppose each row is one participant and each column is one trial-wise accuracy value.
trial_acc = np.array([
    [0.92, 0.95, 0.91, 0.94],
    [0.81, 0.79, 0.84, 0.80],
    [0.88, 0.90, 0.87, 0.89]
])

# Task:
# 1) Compute the row means with keepdims=True.
# 2) Subtract the row mean from each row.
# 3) Verify that the centered rows have mean approximately 0.

row_means = np.mean(trial_acc, axis=1, keepdims=True)
centered = trial_acc - row_means

print("row means shape:", row_means.shape)
print(centered)
print(np.mean(centered, axis=1))


## 15. Challenge: identify high-performing memory participants

Build a final filtered table with these steps:

1. Start from `df_filled` if you already created it. Otherwise create it again.
2. Keep only rows where:
   - `task == "memory"`
   - `accuracy >= 0.85`
3. Create a new column `speed_score = 1000 / rt_ms`
4. Sort by `speed_score` descending
5. Return only:
   - `participant_id`
   - `group`
   - `accuracy`
   - `rt_ms`
   - `speed_score`

Try to write this as a compact pandas pipeline.


In [ ]:
# Starter code:

df_filled = df.copy()
df_filled["accuracy"] = df_filled["accuracy"].fillna(df_filled["accuracy"].mean())

result = (
    df_filled
    [(df_filled["task"] == "memory") & (df_filled["accuracy"] >= 0.85)]
    .assign(speed_score=lambda data: 1000 / data["rt_ms"])
    .sort_values(by="speed_score", ascending=False)
    [["participant_id", "group", "accuracy", "rt_ms", "speed_score"]]
)

result